In [ ]:
import pickle
import json
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt

import requests
from PIL import Image

In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
root_override = os.environ.get("OCL_PROJECT_ROOT")
candidates = [Path(root_override).resolve()] if root_override else [cwd, *cwd.parents]
project_root = next((p for p in candidates if (p / "src/dataset.py").is_file()), None)
if project_root is None:
    raise RuntimeError("Open the notebook from the OCL workspace or set OCL_PROJECT_ROOT.")

sys.path.insert(0, str(project_root))
data_root = Path(os.environ.get("OCL_DATA_ROOT", project_root / "data"))
resource_root = Path(os.environ.get("OCL_RESOURCE_ROOT", data_root / "resources"))



with open(resource_root / "OCL_annot_test.pkl", "rb") as f:
    test_data = pickle.load(f)

def load_json(filename):
    with open(resource_root / filename, "r", encoding="utf-8") as f:
        return json.load(f)

categories = load_json("OCL_class_object.json")
attributes = load_json("OCL_class_attribute.json")
affordances = load_json("OCL_class_affordance.json")

print("Resource path:", resource_root)
print("Number of images:", len(test_data))
print("Number of categories:", len(categories))
print("Number of attributes:", len(attributes))
print("Number of affordances:", len(affordances))

In [ ]:
image = test_data[0]
obj = image["objects"][0]

print("Image path:", image["name"])
print("Source:", image["source"])
print("Category:", obj["obj"])
print("Bounding box:", obj["box"])

print("\nAttributes:")
print([attributes[i] for i in obj["attr"]])

print("\nAffordances:")
print([affordances[i]["word"][0] for i in obj["aff"]])

In [ ]:
category_counts = Counter()

for image in test_data:
    for obj in image["objects"]:
        category = obj["obj"]
        category_counts[category] += 1

top10 = category_counts.most_common(10)

print("Total objects:", sum(category_counts.values()))
print("Unique categories:", len(category_counts))

for category, count in top10:
    print(category, count)

In [ ]:
names = [item[0] for item in top10]
counts = [item[1] for item in top10]

plt.figure(figsize=(10, 5))
plt.barh(names, counts)

plt.gca().invert_yaxis()

plt.xlabel("Number of objects")
plt.ylabel("Category")
plt.title("Top 10 Object Categories in OCL Test Set")

plt.tight_layout()
plt.show()

In [ ]:
top10_categories = {item[0] for item in top10}

samples = []

for image in test_data:
    for obj in image["objects"]:

        if obj["obj"] in top10_categories:
            sample = {
                "name": image["name"],
                "source": image["source"],
                "category": obj["obj"],
                "box": obj.get("box"),
                "attr": obj["attr"],
                "aff": obj["aff"]
            }

            samples.append(sample)

print("Number of selected objects:", len(samples))
print(samples[0])

In [ ]:
source_object_counts = Counter()
source_images = {}

for sample in samples:
    source = sample["source"]
    name = sample["name"]
    source_object_counts[source] += 1
    
    if source not in source_images:
        source_images[source] = set()

    source_images[source].add(name)


for source in source_object_counts:
    print(
        source,
        "Objects:", source_object_counts[source],
        "Images:", len(source_images[source])
    )

In [ ]:
assert len(samples) == sum(count for _, count in top10)

assert all(
    sample["category"] in top10_categories
    for sample in samples
)

assert all(
    0 <= i < len(attributes)
    for sample in samples
    for i in sample["attr"]
)

assert all(
    0 <= i < len(affordances)
    for sample in samples
    for i in sample["aff"]
)


from the top,we know we only need to get COCO/ImageNet/from_web

In [ ]:
manifest = {source: sorted(names) for source, names in source_images.items()}
manifest_path = data_root / "image_manifest.json"
if manifest_path.is_file():
    saved = json.loads(manifest_path.read_text(encoding="utf-8-sig"))
    if saved != manifest:
        raise ValueError("The existing manifest differs. Use a separate data root to generate a new one.")
    print("Existing manifest verified:", manifest_path)
else:
    data_root.mkdir(parents=True, exist_ok=True)
    with manifest_path.open("x", encoding="utf-8") as stream:
        json.dump(manifest, stream, indent=2)
    print("Manifest saved:", manifest_path)

In [ ]:
for source, names in manifest.items():
    print(f"\n{source}:")

    for name in names[:5]:
        print(name)

In [ ]:
coco_folders = Counter(
    str(Path(name).parent)
    for name in manifest["COCO"]
)

print(coco_folders)

In [ ]:
url_mapping = load_json("url_to_filename.json")

print(type(url_mapping))
print(len(url_mapping))
print(url_mapping[:3])

In [ ]:
from io import BytesIO

name = manifest["COCO"][0]
image_path = data_root / name
if image_path.is_file():
    image_bytes = image_path.read_bytes()
    print("Using the local image:", image_path)
else:
    split, filename = image_path.parent.name, image_path.name
    url = f"https://images.cocodataset.org/{split}/{filename}"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    image_bytes = response.content
    print("Loaded for display only:", url)

In [ ]:
image = Image.open(BytesIO(image_bytes))

print(image.size)
print(image.mode)

display(image)

In [ ]:
web_url_dict = {
    filename: url
    for url, filename in url_mapping
}

In [ ]:
web_available = []
web_missing = []

for name in manifest["from_web"]:
    if name in web_url_dict:
        web_available.append(name)
    else:
        web_missing.append(name)

print("Total:", len(manifest["from_web"]))
print("URL available:", len(web_available))
print("URL missing:", len(web_missing))

print(web_missing[:5])